# Análisis multivariante aplicado a la tabla final del TFM

Este notebook adapta la estructura del notebook de la asignatura de técnicas multivariantes al caso del TFM. A diferencia del ejercicio de regresión, aquí no existe una variable respuesta: el objetivo es identificar perfiles territoriales latentes entre los distritos de Valencia. Por ello, el flujo se centra en análisis descriptivo, preparación de la matriz, PCA, K-Means, validación y lectura territorial de resultados.

In [4]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler

RANDOM_STATE = 123
METADATA_CODES = {"__categoria__", "__subdimension__", "__rol_analisis__"}

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "02_datos_finales").exists():
            return candidate
    raise FileNotFoundError("No se encontro la carpeta 02_datos_finales del repositorio.")

PROJECT_ROOT = find_project_root()
ANALYSIS_DIR = PROJECT_ROOT / "análisis_multivariante_17junio"
DATA_DIR = PROJECT_ROOT / "02_datos_finales"
RESULTS_DIR = ANALYSIS_DIR / "resultados"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TABLE_PATH = DATA_DIR / "tabla_por_distritos_final_jerarquizada.csv"
DICTIONARY_PATH = DATA_DIR / "diccionario_jerarquia_variables_final.csv"

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

## 1. Carga de datos

Se parte de la tabla final jerarquizada y del diccionario de variables. La tabla contiene filas de metadatos (`__categoria__`, `__subdimension__`, `__rol_analisis__`) que deben excluirse antes de aplicar PCA o clustering.

In [ ]:
tabla = pd.read_csv(TABLE_PATH, dtype=str)
diccionario = pd.read_csv(DICTIONARY_PATH, dtype=str)

distritos = tabla[~tabla["codigo"].isin(METADATA_CODES)].copy().reset_index(drop=True)

print("Filas de distritos:", distritos.shape[0])
print("Columnas totales:", distritos.shape[1])
distritos[["codigo", "nombre"]].head()

NameError: name 'pd' is not defined

## 2. Preparación de la matriz activa

Se seleccionan las variables marcadas como `Activa candidata` en el diccionario. Para la solución principal se utiliza solo el subconjunto sin valores perdidos, evitando imputar datos en el ajuste principal.

In [ ]:
def parse_number_series(series: pd.Series) -> pd.Series:
    clean = (
        series.astype(str)
        .str.strip()
        .str.replace("\u00a0", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(clean, errors="coerce")

active_variables = [
    variable for variable in diccionario.loc[diccionario["rol_analisis"].eq("Activa candidata"), "variable"]
    if variable in distritos.columns
]

X_active = pd.DataFrame({variable: parse_number_series(distritos[variable]) for variable in active_variables})

quality = pd.DataFrame({
    "variable": active_variables,
    "n_missing": [int(X_active[c].isna().sum()) for c in active_variables],
    "n_observations": [int(X_active[c].notna().sum()) for c in active_variables],
    "n_unique": [int(X_active[c].nunique(dropna=True)) for c in active_variables],
})
quality = quality.merge(diccionario[["variable", "bloque_tematico", "subdimension", "rol_analisis"]], on="variable", how="left")

valid_variables = quality.query("n_observations >= 15 and n_unique > 1")["variable"].tolist()
complete_variables = quality.query("n_observations >= 15 and n_unique > 1 and n_missing == 0")["variable"].tolist()

print("Variables activas candidatas:", len(active_variables))
print("Variables validas con al menos 15 observaciones:", len(valid_variables))
print("Variables completas usadas en la solucion principal:", len(complete_variables))
quality.sort_values("n_missing", ascending=False).head(12)

## 3. Escalado MinMax

El escalado es necesario porque la matriz combina porcentajes, tasas por población, densidades, valores monetarios e índices. Se utiliza `MinMaxScaler` para llevar todas las variables al intervalo `[0, 1]`, preservando el orden relativo entre distritos.

In [ ]:
X_complete = X_active[complete_variables].copy()
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_complete)

pd.DataFrame(X_scaled, columns=complete_variables).describe().T.head()

## 4. PCA exploratorio

El PCA no sustituye al clustering, pero ayuda a comprobar si existe estructura multivariante y a visualizar la posición relativa de los distritos en las primeras componentes.

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

pca_var = pd.DataFrame({
    "componente": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "varianza_explicada": pca.explained_variance_ratio_,
    "varianza_acumulada": np.cumsum(pca.explained_variance_ratio_),
})
pca_var.head(10)

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(range(1, 11), pca_var["varianza_acumulada"].head(10), marker="o")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada")
plt.title("Varianza explicada acumulada por PCA")
plt.grid(alpha=0.3)
plt.show()

## 5. Elección del número de clusters

Se calcula K-Means para distintos valores de `k`. La decisión no depende de un único indicador: se combinan inercia, silhouette, Davies-Bouldin, Calinski-Harabasz e interpretabilidad territorial.

In [ ]:
def kmeans_metrics(X, k_values=range(2, 9)):
    rows = []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=100)
        labels = model.fit_predict(X)
        rows.append({
            "k": k,
            "inercia": model.inertia_,
            "silhouette": silhouette_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels),
            "tamano_clusters": ",".join(str(v) for v in np.bincount(labels)),
        })
    return pd.DataFrame(rows)

metricas = kmeans_metrics(X_scaled)
metricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].plot(metricas["k"], metricas["inercia"], marker="o")
axes[0].set_title("Método del codo")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inercia")

axes[1].plot(metricas["k"], metricas["silhouette"], marker="o", color="#2a9d8f")
axes[1].set_title("Silhouette")
axes[1].set_xlabel("k")

axes[2].plot(metricas["k"], metricas["davies_bouldin"], marker="o", color="#e76f51")
axes[2].set_title("Davies-Bouldin")
axes[2].set_xlabel("k")

plt.tight_layout()
plt.show()

## 6. Ajuste final con k = 4

La solución con `k=4` reproduce una tipología territorial interpretable: un grupo amplio de distritos compactos intermedios, Ciutat Vella como centro histórico singular, el eje L'Eixample-Extramurs-Pla del Real y los Poblats como periferia extensiva.

In [ ]:
modelo_k4 = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=100)
labels_k4 = modelo_k4.fit_predict(X_scaled)

clusters = distritos[["codigo", "nombre"]].copy()
clusters["cluster"] = labels_k4
clusters["distancia_centroide"] = np.linalg.norm(X_scaled - modelo_k4.cluster_centers_[labels_k4], axis=1)
clusters["representante_cluster"] = False

for cluster in sorted(clusters["cluster"].unique()):
    idx = clusters.index[clusters["cluster"].eq(cluster)].to_numpy()
    representante = idx[np.argmin(clusters.loc[idx, "distancia_centroide"].to_numpy())]
    clusters.loc[representante, "representante_cluster"] = True

clusters.sort_values(["cluster", "distancia_centroide"])

In [ ]:
plt.figure(figsize=(7.5, 5.8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_k4, cmap="tab10", s=80, edgecolor="white")
for i, nombre in enumerate(distritos["nombre"]):
    plt.text(X_pca[i, 0], X_pca[i, 1], nombre, fontsize=7)
plt.xlabel(f"PC1 ({pca_var.loc[0, 'varianza_explicada']:.1%})")
plt.ylabel(f"PC2 ({pca_var.loc[1, 'varianza_explicada']:.1%})")
plt.title("Distritos proyectados sobre las dos primeras componentes")
plt.show()

## 7. Perfil de clusters en variables interpretativas

Para redactar resultados no basta con nombrar el distrito más cercano al centroide. Conviene interpretar los grupos mediante variables sintéticas legibles: renta, densidad, paro, pobreza infantil, vivienda turística, actividad económica, zonas verdes o hechos discriminatorios.

In [ ]:
interpretative_variables = [
    "Demografia - Densidad poblacion [hab_km2]",
    "Economia - Renta neta media por persona [EUR]",
    "Economia - Poblacion parada [% 16+]",
    "Pobreza - Infantil (2021) Tasa distrito [%]",
    "Vivienda - Valor catastral medio por m2 [EUR_m2]",
    "Economia - Actividades economicas [por_1000_hab]",
    "Economia - Viviendas turisticas [% viviendas]",
    "Demografia - Hogares unipersonales [%]",
    "Medio ambiente - Superficie zonas verdes [m2_hab]",
    "Hechos discriminatorios [por_1000_hab]",
]

perfil = X_active[interpretative_variables].copy()
perfil["cluster"] = labels_k4
perfil_clusters = perfil.groupby("cluster")[interpretative_variables].mean().round(2)
perfil_clusters

## 8. Variables más y menos influyentes

Como medida descriptiva de separación entre grupos se calcula eta cuadrado: proporción de variación de cada variable explicada por las diferencias entre clusters. No es un test causal; sirve para ordenar variables según su capacidad discriminante dentro de la partición obtenida.

In [ ]:
def eta_squared_by_variable(X, columns, labels):
    frame = pd.DataFrame(X, columns=columns)
    overall = frame.mean(axis=0)
    ss_total = ((frame - overall) ** 2).sum(axis=0)
    ss_between = pd.Series(0.0, index=columns)

    for cluster in sorted(set(labels)):
        mask = labels == cluster
        cluster_mean = frame.loc[mask].mean(axis=0)
        ss_between += int(mask.sum()) * ((cluster_mean - overall) ** 2)

    return (ss_between / ss_total.replace(0, np.nan)).sort_values(ascending=False)

eta = eta_squared_by_variable(X_scaled, complete_variables, labels_k4).reset_index()
eta.columns = ["variable", "eta2"]
eta = eta.merge(diccionario[["variable", "bloque_tematico", "subdimension"]], on="variable", how="left")
eta.head(15)

In [ ]:
eta.tail(15)

## 9. Robustez: comparación k = 3, k = 4 y k = 5

La comparación de soluciones sirve para justificar por qué `k=4` ofrece un equilibrio razonable. Con `k=3`, Ciutat Vella deja de formar un grupo propio; con `k=5`, el grupo intermedio se subdivide y la lectura pierde parsimonia.

In [ ]:
comparacion = distritos[["codigo", "nombre"]].copy()
for k in [3, 4, 5]:
    modelo = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=100)
    comparacion[f"cluster_k{k}"] = modelo.fit_predict(X_scaled)
comparacion

## 10. Exportación de resultados

Se guardan las tablas principales para que el texto del TFM no dependa de cálculos manuales.

In [ ]:
metricas.to_csv(RESULTS_DIR / "metricas_kmeans_variables_activas_completas.csv", index=False)
clusters.sort_values(["cluster", "distancia_centroide"]).to_csv(RESULTS_DIR / "clusters_k4_variables_activas_completas.csv", index=False)
perfil_clusters.to_csv(RESULTS_DIR / "perfil_clusters_k4_variables_interpretativas_desde_notebook.csv")
eta.to_csv(RESULTS_DIR / "variables_mas_menos_influyentes_k4.csv", index=False)
pca_var.to_csv(RESULTS_DIR / "pca_varianza_explicada.csv", index=False)
comparacion.to_csv(RESULTS_DIR / "comparacion_clusters_k3_k4_k5.csv", index=False)

print("Resultados guardados en", RESULTS_DIR)

## Lectura para el TFM

La solución con `k=4` debe presentarse como una tipología exploratoria e interpretable. Los indicadores de validación muestran separación moderada, no una frontera rígida entre grupos. La fortaleza del resultado está en la coherencia territorial: Ciutat Vella aparece como centro histórico singular, L'Eixample-Extramurs-Pla del Real como eje central acomodado, los Poblats como periferia extensiva y el resto como ciudad compacta intermedia.